# PyTorch Autograd

## 학습 내용
- requires_grad를 이용한 gradient 추적
- grad_fn과 Computational Graph
- backward()를 이용한 역전파
- gradient 계산 및 .grad 저장
- gradient 누적(Accumulation)
- zero_grad()를 이용한 gradient 초기화

In [ ]:
import torch

x = torch.ones(5)
y = torch.zeros(3)
# 가중치(weight), 학습 대상이므로 gradient 추적
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
# 선형 연산 수행 (z = xW + b)
z = torch.matmul(x, w)+b
# 예측값(z)과 정답(y)을 비교하여 손실(loss) 계산
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

In [ ]:
# grad_fn은 이 텐서를 만든 연산 정보

# z가 어떤 연산을 통해 생성되었는지 확인
print(f" Gradient function for z = {z.grad_fn}")

# loss가 어떤 연산을 통해 생성되었는지 확인
print(f"Gradient function for loss = {loss.grad_fn}")

 Gradient function for z = <AddBackward0 object at 0x00000125BA0DFA90>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x00000125BA0DFCD0>


In [ ]:
# 역전파
# 손실(loss)을 기준으로 w, b의 기울기를 계산
loss.backward()
print(w.grad)
print(b.grad)

tensor([[0.1597, 0.2781, 0.2883],
        [0.1597, 0.2781, 0.2883],
        [0.1597, 0.2781, 0.2883],
        [0.1597, 0.2781, 0.2883],
        [0.1597, 0.2781, 0.2883]])
tensor([0.1597, 0.2781, 0.2883])


In [ ]:
# gradient 추적 활성화
z = torch.matmul(x, w) + b
print(z.requires_grad)

# gradient 추적 비활성화
with torch.no_grad():
    z = torch.matmul(x, w) + b
print(z.requires_grad)

# 모델 평가 및 추론 때 활용

True
False


In [ ]:
z = torch.matmul(x, w) + b
# 계산 그래프로부터 분리하여 gradient 추적 비활성화
z_det = z.detach()
print(z_det.requires_grad)

# z_det은 역전파(backpropagation)에 참여하지 않음

False


## Computational Graph

- PyTorch는 연산 과정과 결과를 Computational Graph(DAG)로 저장한다.
- Forward Pass에서 연산을 수행하면서 동시에 계산 그래프를 생성한다.
- 각 Tensor는 자신을 생성한 연산 정보를 grad_fn에 저장한다.
- backward() 호출 시 계산 그래프를 역방향으로 따라가며 gradient를 계산한다.
- 계산된 gradient는 각 Tensor의 .grad 속성에 저장된다.
- Gradient 계산에는 Chain Rule(연쇄 법칙)이 사용된다.
- PyTorch의 계산 그래프는 Dynamic Graph 방식으로 매 반복마다 새로 생성된다.

In [ ]:
# [4, 5] 행렬 생성
inp = torch.eye(4, 5, requires_grad=True)
# 모든 원소에 1을 더함
# .pow(2) 제곱
# .t() 전치행렬
out = (inp + 1).pow(2).t()

# 첫 번째 역전파 수행
# out은 행렬이므로 각 원소의 gradient를 1로 지정
# retain_graph=True는 게산 그래프를 유지하여 다시 backward를 가능하게 함
out.backward(torch.ones_like(out), retain_graph=True)

# 첫 번째 backward 계산 출력
print(f"First call\n{inp.grad}")

# 두 번째 역전파 수행
# gradient는 덮어쓰지 않고 기존 값에 누적됨
out.backward(torch.ones_like(out), retain_graph=True)

# 누적된 gradient 출력 (첫 번째 결과의 2배)
print(f"Second call\n{inp.grad}")

# gradient를 0으로 초기화
inp.grad.zero_()

# 다시 역전파 수행
out.backward(torch.ones_like(out), retain_graph=True)

# gradient 초기화 후 다시 계산된 결과 출력
print(f"\nCall after zeroing gradients\n{inp.grad}")

# 한 배치(64개 데이터)의 gradient를 계산한 뒤 weight를 업데이트한다.
# PyTorch는 gradient를 누적하므로 다음 배치를 학습하기 전에 zero_grad()로 gradient를 초기화한다.

First call
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])
Second call
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])

Call after zeroing gradients
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])


## 정리
- Pytorch는 backward() 호출 시 gradient를 .grad에 누적한다.
- 따라서 여러 번 backward()를 호출하면 gradient가 계속 더해진다.
- 학습 시 현재 배치의 gradient만 사용하기 위해 이전 gradient를 초기화해야 한다.
- gradient 초기화는 optimizer.zero_grad() 또는 grad.zero_()를 사용한다.
- 학습 루프에서는 보통 `zero_grad() → backward() → step()` 순서로 진행한다.